In [16]:
import numpy as np
import scipy.stats
import scipy.special
import matplotlib.pyplot as plt
import random
import yaml
import os
import pickle
import torch
import sbi
from sbi.inference import NPE_C
import pickle
from examples.lgssm import make_prior, simulator
results_path = "/Users/Lieve/Documents/Masters Project/lgssm_results_from_cluster_partial/"

In [29]:
algorithm_names = ["npe_c", "tsnpe", "nle_mcmc", "nle_vi", "nre"]
algorithm_names_to_nice_sequential = {"npe_c": "SNPE-C (multi-round)",
                                  "tsnpe": "TSNPE (multi-round)",
                                  "nle_mcmc": "SNLE-MCMC (multi-round)",
                                  "nle_vi": "SNLE-VI (multi-round)",
                                  "nre": "SNRE-B (multi-round)"}
algorithm_names_to_nice_amortized = {"npe_c": "SNPE-C (single-round)",
                                  "tsnpe": "TSNPE (single-round)",
                                  "nle_mcmc": "SNLE-MCMC (single-round)",
                                  "nle_vi": "SNLE-VI (single-round)",
                                  "nre": "SNRE-B (single-round)"}
algorithm_names = ["nle_vi", "tsnpe", "npe_c"]
T_list = [10,25,50,100]
method_type_list = ["amortized", "sequential"]
x_observed_ID = 0

In [ ]:
# Import trajectory 
trajectory_ID = x_observed_ID
trajectory_path = results_path + f"data/trajectory{trajectory_ID}.npz"
trajectory_config_path = results_path + f"data/trajectory{trajectory_ID}.yaml"

trajectory = np.load(trajectory_path)

with open(trajectory_config_path, "r") as f:
    trajectory_config = yaml.safe_load(f)
rho_true = trajectory_config["rho_true"]
sigma_true = trajectory_config["sigma_true"]
tau_true = trajectory_config["tau_true"]
trajectory_config

In [30]:
rho_means = {name: {method_type: {T: [] for T in T_list} for method_type in method_type_list} for name in algorithm_names}
rho_lqs = {name: {method_type: {T: [] for T in T_list} for method_type in method_type_list} for name in algorithm_names}
rho_uqs = {name: {method_type: {T: [] for T in T_list} for method_type in method_type_list} for name in algorithm_names}

tau_means = {name: {method_type: {T: [] for T in T_list} for method_type in method_type_list} for name in algorithm_names}
tau_lqs = {name: {method_type: {T: [] for T in T_list} for method_type in method_type_list} for name in algorithm_names}
tau_uqs = {name: {method_type: {T: [] for T in T_list} for method_type in method_type_list} for name in algorithm_names}

In [32]:
num_samples = 2500 # Number of samples to take from each posterior

for method_type in method_type_list:
    for algorithm_name in algorithm_names:
        print(f"{method_type} {algorithm_name}")
        for T in T_list:
            print(f"T = {T}:")
            for i in range(10): # Number of repeated experiments
                path = results_path + f"{algorithm_name}" + f"/{method_type}_posterior_T{T}_xobsid{x_observed_ID}__{i}_posteriors_dict.pkl"
                if os.path.exists(path):
                    # Load pickle file
                    with open(path, "rb") as f:
                        posteriors_dict = pickle.load(f)
                    if method_type == "amortized":
                        final_round_posterior = posteriors_dict["round_0"]
                    else:
                        final_round_posterior = posteriors_dict["round_7"]
                    
                    if algorithm_name in ["nle_mcmc"]:
                        samples = final_round_posterior.sample((num_samples//2,), num_chains=2)
                    elif algorithm_name in ["nle_vi"]:
                        #x_observed = trajectory["x"][:T+1]
                        #x_observed = torch.tensor(x_observed)
                        print("Training VI posterior:")
                        final_round_posterior.train()
                        print("VI posterior trained.")
                        samples = final_round_posterior.sample((num_samples,))
                    else:
                        samples = final_round_posterior.sample((num_samples,))


                    rho_samples = samples[:, 0]
                    tau_samples = samples[:, 1]

                    # Means
                    rho_mean = torch.mean(rho_samples).item()
                    tau_mean = torch.mean(tau_samples).item()
                    rho_means[algorithm_name][method_type][T].append(rho_mean)
                    tau_means[algorithm_name][method_type][T].append(tau_mean)

                    # Lower quartiles
                    rho_lq = torch.quantile(rho_samples, 0.025).item()
                    tau_lq = torch.quantile(tau_samples, 0.025).item()
                    rho_lqs[algorithm_name][method_type][T].append(rho_lq)
                    tau_lqs[algorithm_name][method_type][T].append(tau_lq)

                    # Upper quartiles
                    rho_uq = torch.quantile(rho_samples, 0.975).item()
                    tau_uq = torch.quantile(tau_samples, 0.975).item()
                    rho_uqs[algorithm_name][method_type][T].append(rho_uq)
                    tau_uqs[algorithm_name][method_type][T].append(tau_uq)

amortized nle_vi
T = 10:
Training VI posterior:


  0%|          | 0/2000 [00:00<?, ?it/s]

/Users/Lieve/Documents/Masters Project/sbi_venv/lib/python3.9/site-packages/sbi/inference/posteriors/vi_posterior.py:511: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  metric = round(float(quality_control_fn(self, N=N)), 3)


Quality Score: 0.007 	 Good: Smaller than 0.5  Bad: Larger than 1.0 	         NOTE: Less sensitive to mode collapse.
VI posterior trained.


TimeoutError: [Errno 60] Operation timed out

In [ ]:
rho_means_mean = {name: {method_type: {} for method_type in method_type_list} for name in algorithm_names}
rho_means_std = {name: {method_type: {} for method_type in method_type_list} for name in algorithm_names}

rho_lqs_mean = {name: {method_type: {} for method_type in method_type_list} for name in algorithm_names}
rho_lqs_std = {name: {method_type: {} for method_type in method_type_list} for name in algorithm_names}

rho_uqs_mean = {name: {method_type: {} for method_type in method_type_list} for name in algorithm_names}
rho_uqs_std = {name: {method_type: {} for method_type in method_type_list} for name in algorithm_names}

tau_means_mean = {name: {method_type: {} for method_type in method_type_list} for name in algorithm_names}
tau_means_std = {name: {method_type: {} for method_type in method_type_list} for name in algorithm_names}

tau_lqs_mean = {name: {method_type: {} for method_type in method_type_list} for name in algorithm_names}
tau_lqs_std = {name: {method_type: {} for method_type in method_type_list} for name in algorithm_names}

tau_uqs_mean = {name: {method_type: {} for method_type in method_type_list} for name in algorithm_names}
tau_uqs_std = {name: {method_type: {} for method_type in method_type_list} for name in algorithm_names}

In [ ]:
for method_type in method_type_list:
    for algorithm_name in algorithm_names:
        for T in T_list:
            # Number of repeated samples
            n = len(rho_means[algorithm_name][method_type][T])
            if n > 1: # Need at least 2 samples to compute std
                rho_means_mean[algorithm_name][method_type][T] = np.mean(rho_means[algorithm_name][method_type][T])
                rho_means_std[algorithm_name][method_type][T] = np.std(rho_means[algorithm_name][method_type][T])
                
                rho_lqs_mean[algorithm_name][method_type][T] = np.mean(rho_lqs[algorithm_name][method_type][T])
                rho_lqs_std[algorithm_name][method_type][T] = np.std(rho_lqs[algorithm_name][method_type][T])

                rho_uqs_mean[algorithm_name][method_type][T] = np.mean(rho_uqs[algorithm_name][method_type][T])
                rho_uqs_std[algorithm_name][method_type][T] = np.std(rho_uqs[algorithm_name][method_type][T])

                tau_means_mean[algorithm_name][method_type][T] = np.mean(tau_means[algorithm_name][method_type][T])
                tau_means_std[algorithm_name][method_type][T] = np.std(tau_means[algorithm_name][method_type][T])
                
                tau_lqs_mean[algorithm_name][method_type][T] = np.mean(tau_lqs[algorithm_name][method_type][T])
                tau_lqs_std[algorithm_name][method_type][T] = np.std(tau_lqs[algorithm_name][method_type][T])

                tau_uqs_mean[algorithm_name][method_type][T] = np.mean(tau_uqs[algorithm_name][method_type][T])
                tau_uqs_std[algorithm_name][method_type][T] = np.std(tau_uqs[algorithm_name][method_type][T])